Duilio Lucio, Vivian Hu

Spring 2026

CS343: Neural Networks

Project 3: Convolutional Neural Networks

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.rcParams.update({'font.size': 20})

np.set_printoptions(suppress=True, precision=7)

# Automatically reload your external source code
%load_ext autoreload
%autoreload 2

## Goal

The goal of this notebook is to walk you through the recommended implementation order for your convolutional neural network and provide test code to help make sure your implementation works as expected every step of the way.

Next week, you will test your network on the STL-10 dataset.

**Global note: Make sure any debug printouts do not appear if `verbose=False`!**


## Task 4: Building a convolutional neural network

Now that you have the core convolution and max pool operations implemented, you can tackle the main task of building a convolutional neural network. This will be a "deep" 4-layer neural network with the following architecture:

1. Convolution (net-in), Relu (net-act).
2. Max pool 2D (net-in), linear (net-act).
3. Flatten (net-in), linear (net-act).
4. Dense (net-in), Relu (net-act).
5. Dense (net-in), soft-max (net-act).

In the above outline, the first part is the layer net-in type (e.g. conv, maxpool, etc) and the second is the layer's activation function (rectified linear (Relu), soft-max, etc). 

Unlike the MLP project, your network will adopt an object-oriented, modular design that should make it straightforward to add/remove/customize layers with minimal code changes.

### 4a. Migrate existing code (`layer.py`)

Copy-paste your code from the last project to implement the following functions:

- `one_hot`

### 4b. Network layer activation functions (`layer.py`)

Implement the following activation functions. Remember, an activation function transforms a layer's "net input" to "net activation".

- `linear`
- `relu`
- `softmax`
- `compute_net_act`

Equation for softmax:

$e^{x_{ij}} / \sum_{k=1}^C e^{x_{ik}}$ where $x_{ij}$ is the "net in" value for neuron $j$ for input $i$. $C$ corresponds to the number of classes in the dataset.

In [5]:
from layer import *

#### Test: `linear()`

In [6]:
test_layer = Layer(0, 'test')
test_layer.net_in = np.arange(10)
test_layer.linear()
print(f'{test_layer.net_act} should be [0 1 2 3 4 5 6 7 8 9]')

[0 1 2 3 4 5 6 7 8 9] should be [0 1 2 3 4 5 6 7 8 9]


#### Test: `relu()`

In [7]:
rng = np.random.default_rng(0)
test_layer = Layer(0, 'test')
test_layer.net_in = rng.random([3, 3]) - 0.5
test_layer.relu()
print(f'{test_layer.net_act}')

[[0.1369617 0.        0.       ]
 [0.        0.3132702 0.4127556]
 [0.1066358 0.2294966 0.043625 ]]


You should get:


    [[0.1369617 0.        0.       ]
     [0.        0.3132702 0.4127556]
     [0.1066358 0.2294966 0.043625 ]]

#### Test: `softmax()`

In [8]:
rng = np.random.default_rng(0)
test_layer = Layer(0, 'test')
test_layer.net_in = rng.random([2, 5])
test_layer.softmax()
print(f'{test_layer.net_act}')

[[0.2516215 0.1742953 0.1386479 0.1352996 0.3001356]
 [0.2334946 0.1719217 0.1943965 0.161423  0.2387641]]


You should get:

    [[0.2516215 0.1742953 0.1386479 0.1352996 0.3001356]
     [0.2334946 0.1719217 0.1943965 0.161423  0.2387641]]

### 4c. Implement loss function

In `layer.py`, implement cross-entropy loss `cross_entropy()` (see `loss()` for usage). 

Mathematical equation:

$-\frac{1}{B}\sum_{i=1}^B Log \left ( y_{ic} \right )$ 

where $y_{ic}$ is the softmax activation value $y$ for the NEURON CODING THE CORRECT CLASS $c$ for the $i^{th}$ input in the mini-batch ($i: 1...B$).

#### Test: `cross_entropy()`

In [13]:
rng = np.random.default_rng(0)
y = np.array([0, 4, 1])
test_layer = Layer(0, 'test')
test_layer.net_in = rng.random([3, 5])
test_layer.softmax()
print(f'Your loss is {test_layer.cross_entropy(y):.5f} and it should be 1.65869')

Your loss is 1.65869 and it should be 1.65869


### 4d. Implement the forward pass of the convolution layer

Do this first because this is the first layer of the network (see above architecture).

Implement and test the following methods in `layer.py`:

- constructor in `Conv2D`
- `compute_net_in` in `Conv2D`
- `forward` in `Layer`. The `forward` method synthesizes your work so far and computes all the forward operations for this (and any other layers you create later on).

##### Test Conv2D initialization

In [17]:
conv2_layer = Conv2D(0, 'conv2', n_kers=2, ker_sz=2, wt_scale=1e-1, r_seed=2)
print(f'Your filter weights are\n{conv2_layer.wts}')
print(f'Your bias terms are\n{conv2_layer.b}')

Your filter weights are
[[[[ 0.0189053 -0.0522748]
   [-0.0413064 -0.2441467]]

  [[ 0.1799707  0.1144166]
   [-0.0325423  0.0773807]]

  [[ 0.0281211 -0.0553823]
   [ 0.0977567 -0.0310557]]]


 [[[-0.0328824 -0.0792147]
   [ 0.0454958 -0.0099198]]

  [[ 0.0545289 -0.0607186]
   [ 0.0126828 -0.0892274]]

  [[ 0.0841465  0.0188035]
   [ 0.0330571  0.0410504]]]]
Your bias terms are
[0. 0.]


The above should yield:

```
Your filter weights are
[[[[ 0.0189053 -0.0522748]
   [-0.0413064 -0.2441467]]

  [[ 0.1799707  0.1144166]
   [-0.0325423  0.0773807]]

  [[ 0.0281211 -0.0553823]
   [ 0.0977567 -0.0310557]]]


 [[[-0.0328824 -0.0792147]
   [ 0.0454958 -0.0099198]]

  [[ 0.0545289 -0.0607186]
   [ 0.0126828 -0.0892274]]

  [[ 0.0841465  0.0188035]
   [ 0.0330571  0.0410504]]]]
Your bias terms are
[0. 0.]
```

#####  Test `forward` using `Conv2D` layer

In [18]:
rng = np.random.default_rng(1)
# Create test net parameters
mini_batch_sz, n_kers, n_chans, ker_sz, img_y, img_x = 1, 2, 3, 4, 5, 5
# Create random test input
inputs = rng.standard_normal((mini_batch_sz, n_chans, img_y, img_x))

# Create a convolution layer with ReLU activation function
conv_layer = Conv2D(0, 'test', n_kers, ker_sz, n_chans=n_chans, wt_scale=1e-1, activation='relu', r_seed=3)

# Do a forward pass thru the layer
net_act = conv_layer.forward(inputs)

# Extract the computed net values
net_in = conv_layer.net_in
wts = conv_layer.get_wts()
inp = conv_layer.input

print(f'Your input stored in the net has shape: {inp.shape} and it should be (1, 3, 5, 5)')
print(f'Your network wts stored in the net has shape: {wts.shape} and it should be (2, 3, 4, 4)')
print(f'Your network activation has shape: {net_act.shape} and it should be (1, 2, 5, 5)')
print(f'Your net-in has shape: {net_in.shape} and it should be (1, 2, 5, 5)')
print()
print('The first chunk of your filters/weights is:\n', wts[0, 0])
print('The first chunk of your net_in is:\n', net_in[0,0])
print('The first chunk of your net_act is:\n', net_act[0,0])

batch_sz=1, n_chan=3, img_x=5, img_y=5
n_kers=2, n_ker_chans=3, ker_y=4, ker_x=4
Your input stored in the net has shape: (1, 3, 5, 5) and it should be (1, 3, 5, 5)
Your network wts stored in the net has shape: (2, 3, 4, 4) and it should be (2, 3, 4, 4)
Your network activation has shape: (1, 2, 5, 5) and it should be (1, 2, 5, 5)
Your net-in has shape: (1, 2, 5, 5) and it should be (1, 2, 5, 5)

The first chunk of your filters/weights is:
 [[ 0.2040919 -0.2555665  0.0418099 -0.056777 ]
 [-0.0452649 -0.0215597 -0.2019986 -0.0231932]
 [-0.0865213  0.3323     0.0225787 -0.0352631]
 [-0.0281287 -0.0668046 -0.1055151 -0.0390801]]
The first chunk of your net_in is:
 [[-0.3821728 -0.0276198 -0.1975144 -0.3618033  0.3823017]
 [ 0.0069128 -0.489592   0.5613122 -0.8224431  0.6976578]
 [-0.1059418  0.5559063 -0.8748313  0.1992463  1.0941355]
 [-0.0830484 -0.2929336  0.0723614 -1.2549845  0.5915084]
 [ 0.4140173 -0.2115999 -0.7605282 -0.2701102 -0.2675973]]
The first chunk of your net_act is:
 [[0.

The expected output above is:

```
The first chunk of your filters/weights is:
 [[ 0.2040919 -0.2555665  0.0418099 -0.056777 ]
 [-0.0452649 -0.0215597 -0.2019986 -0.0231932]
 [-0.0865213  0.3323     0.0225787 -0.0352631]
 [-0.0281287 -0.0668046 -0.1055151 -0.0390801]]
The first chunk of your net_in is:
 [[-0.3821728 -0.0276198 -0.1975144 -0.3618033  0.3823017]
 [ 0.0069128 -0.489592   0.5613122 -0.8224431  0.6976578]
 [-0.1059418  0.5559063 -0.8748313  0.1992463  1.0941355]
 [-0.0830484 -0.2929336  0.0723614 -1.2549845  0.5915084]
 [ 0.4140173 -0.2115999 -0.7605282 -0.2701102 -0.2675973]]
The first chunk of your net_act is:
 [[0.        0.        0.        0.        0.3823017]
 [0.0069128 0.        0.5613122 0.        0.6976578]
 [0.        0.5559063 0.        0.1992463 1.0941355]
 [0.        0.        0.0723614 0.        0.5915084]
 [0.4140173 0.        0.        0.        0.       ]]
```

### 4e. Implement the forward pass of the max pooling layer

The second layer in the `ConvNet4` architecture is a `MaxPool2D` layer (uses `MaxPool2D` to compute `netIn`) that does a max pooling operation on the output (`netAct`) of the previous layer (`Conv2D`).

Implement and test the following methods:
-  `compute_net_in` in `MaxPool2D`

#####  Test `forward` using `MaxPool2D` layer

In [19]:
rng = np.random.default_rng(0)
# Create test net parameters
mini_batch_sz, n_kers, n_chans, ker_sz, img_y, img_x = 1, 2, 3, 4, 6, 6
# Create random test input
inputs = rng.standard_normal((mini_batch_sz, n_chans, img_y, img_x))

# Create a max pooling layer with default (linear) activation function
pool_layer = MaxPool2D(0, 'pool', pool_size=2, strides=2)

# Do a forward pass thru the layer
net_act = pool_layer.forward(inputs)

# Extract the computed net values
net_in = pool_layer.net_in
wts = pool_layer.wts
inp = pool_layer.input

print(f'Your input stored in the net has shape: {inp.shape} and it should be (1, 3, 6, 6)')
print(f'Your network wts stored is None (as it should be)? {wts is None}')
print(f'Your network activation has shape: {net_act.shape} and it should be (1, 3, 3, 3)')
print(f'Your net in has shape: {net_in.shape} and it should be (1, 3, 3, 3)')
print()
print('The first chunk of your net_in is:\n', net_in[0,0])
print('The first chunk of your net_act is:\n', net_act[0,0])

Your input stored in the net has shape: (1, 3, 6, 6) and it should be (1, 3, 6, 6)
Your network wts stored is None (as it should be)? True
Your network activation has shape: (1, 3, 3, 3) and it should be (1, 3, 3, 3)
Your net in has shape: (1, 3, 3, 3) and it should be (1, 3, 3, 3)

The first chunk of your net_in is:
 [[1.304     0.6404227 0.3615951]
 [1.0425134 1.3664635 0.3515101]
 [0.9034702 0.5408456 0.3553727]]
The first chunk of your net_act is:
 [[1.304     0.6404227 0.3615951]
 [1.0425134 1.3664635 0.3515101]
 [0.9034702 0.5408456 0.3553727]]


The expected output above is:

```
The first chunk of your net_in is:
 [[1.304     0.6404227 0.3615951]
 [1.0425134 1.3664635 0.3515101]
 [0.9034702 0.5408456 0.3553727]]
The first chunk of your net_act is:
 [[1.304     0.6404227 0.3615951]
 [1.0425134 1.3664635 0.3515101]
 [0.9034702 0.5408456 0.3553727]]
```

### 4f. Implement the Flatten layer

The 3rd layer is a "fake layer" that acts as glue to format the activations produced by the last max pooling layer (*2D spatial features*) in a way that is appropriate for subsequent dense layers (*1D features*).

Implement and test method: `compute_net_in` in `Flatten`.

**NOTE:** You will test this soon when you implement the next subtask involving the Dense layer forward pass.

### 4g. Implement the forward pass of the Dense layer

The 4th (hidden) and 5th (output) layers in the `ConvNet4` architecture are ones that use `Dense` `netIn` (these are like the layers in ADALINE/MLP).

Implement and test the following methods:
-  constructor in `Dense`
- `compute_net_in` in `Dense`

##### Test Dense initialization

In [20]:
hidden_layer = Dense(6, 'dense', units=10, n_units_prev_layer=3, wt_scale=1e-1, r_seed=1)
print(f'Your filter weights are\n{hidden_layer.wts}')
print(f'Your bias terms are\n{hidden_layer.b}')

Your filter weights are
[[ 0.000123   0.0298746 -0.0274138 -0.0890592 -0.0454671 -0.0991647
   0.0060144  0.1340215 -0.0492207 -0.0620475]
 [ 0.0489842  0.0356887  0.0105414 -0.0930468 -0.0029252  0.0695303
  -0.1344215 -0.0457616 -0.1901223 -0.1289538]
 [-0.1841735 -0.0235091 -0.1267446  0.0271264  0.0156751 -0.0186931
  -0.251676  -0.0538693 -0.0048501  0.0113309]]
Your bias terms are
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


The above should yield:

    Your filter weights are
    [[ 0.000123   0.0298746 -0.0274138 -0.0890592 -0.0454671 -0.0991647
      0.0060144  0.1340215 -0.0492207 -0.0620475]
    [ 0.0489842  0.0356887  0.0105414 -0.0930468 -0.0029252  0.0695303
      -0.1344215 -0.0457616 -0.1901223 -0.1289538]
    [-0.1841735 -0.0235091 -0.1267446  0.0271264  0.0156751 -0.0186931
      -0.251676  -0.0538693 -0.0048501  0.0113309]]
    Your bias terms are
    [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

#### Test Dense layer forward pass

In [21]:
rng = np.random.default_rng(0)
mini_batch_sz, n_kers, n_chans, ker_sz, img_y, img_x = 2, 2, 3, 4, 6, 6
inputs = rng.standard_normal((mini_batch_sz, n_chans, img_y, img_x))

flat_layer = Flatten(4, 'flatten')
hidden_layer = Dense(5, 'hidden', units=5, wt_scale=1e-1, n_units_prev_layer=n_chans*img_y*img_x, activation='relu', r_seed=2)
hidden_layer.b -= 0.1

net_act_f = flat_layer.forward(inputs)
net_act = hidden_layer.forward(net_act_f)
net_in = hidden_layer.net_in
wts = hidden_layer.wts
inp = hidden_layer.input

print(f'Your flattened layer activations have shape: {net_act_f.shape} and it should be (2, 108)')
print(f'Your input stored in the net has shape: {inp.shape} and it should be (2, 108)')
print(f'Your network wts have shape {wts.shape} and it should be (108, 5)')
print(f'Your network activation has shape: {net_act.shape} and it should be (2, 5)')
print(f'Your net in has shape: {net_in.shape} and it should be (2, 5)')
print()
print('Your net_in is:\n', net_in)
print('Your net_act is:\n', net_act)

Your flattened layer activations have shape: (2, 108) and it should be (2, 108)
Your input stored in the net has shape: (2, 108) and it should be (2, 108)
Your network wts have shape (108, 5) and it should be (108, 5)
Your network activation has shape: (2, 5) and it should be (2, 5)
Your net in has shape: (2, 5) and it should be (2, 5)

Your net_in is:
 [[-0.2454096  0.279862  -1.0326943 -1.4517583 -1.3413267]
 [ 2.2943278 -0.8714551 -0.6093981 -0.6642163 -0.8971514]]
Your net_act is:
 [[0.        0.279862  0.        0.        0.       ]
 [2.2943278 0.        0.        0.        0.       ]]


The expected output above is:

        Your net_in is:
        [[-0.2454096  0.279862  -1.0326943 -1.4517583 -1.3413267]
        [ 2.2943278 -0.8714551 -0.6093981 -0.6642163 -0.8971514]]
        Your net_act is:
        [[0.        0.279862  0.        0.        0.       ]
        [2.2943278 0.        0.        0.        0.       ]]

### 4h. Implement network full forward pass

Now it's time to chain all the individual layers together into a network. 

In `network.py`, implement and the following methods:

- `forward`. This is the forward pass in the that calls each layer's `forward` method that you implemented above. The result of this method will be the loss derived from the activation (`net_act`) of the Output layer, 5 layers deep.
- `wt_reg_reduce`. This is needed for a complete implementation of the full forward method.

Before you can test the forward pass of the network, you need to define what layers belong in the network and how they are arranged! This is done by making a subclass of `Network`. 

- Implement the constructor of `ConvNet4`, adding the layers (in forward pass order):

Conv2D → MaxPool2D → Flatten → Dense → Dense

##### Test network forward pass

In [ ]:
from network import ConvNet4

In [ ]:
rng = np.random.default_rng(3)
n_inputs = 5
X = rng.standard_normal((n_inputs, 3, 32, 32))
y = rng.integers(10, size=n_inputs)

net = ConvNet4(wt_scale=5e-2, r_seed=10)
loss = net.forward(X, y)
print(f'Forward testing loss is {loss:.4f} and it should be 3.9786')

net.reg = 0.1
for layer in net.layers:
    if hasattr(layer, 'reg'):
        layer.reg = 0.1
loss = net.forward(X, y)
print(f'Forward testing regularized loss is {loss:.4f} and it should be 107.1205')
print()
print(f'Your output layer activation values are\n{net.layers[-1].net_act}')

The above should print:

    Your output layer activation values are
    [[0.0525313 0.0186326 0.2586087 0.0731411 0.0144057 0.2968878 0.0379806
      0.012163  0.1634045 0.0722447]
    [0.0422515 0.038249  0.1812459 0.06355   0.0072499 0.1149703 0.2153413
      0.0639554 0.2057011 0.0674856]
    [0.0277646 0.0116104 0.2844209 0.044586  0.02404   0.308795  0.050673
      0.0166541 0.2130068 0.0184493]
    [0.0593645 0.0047332 0.1599501 0.1049977 0.0281754 0.3471514 0.0596684
      0.0121572 0.1878336 0.0359684]
    [0.0374145 0.0445428 0.1307048 0.0643659 0.0261045 0.3312894 0.0545473
      0.0231425 0.2029713 0.0849171]]

### 4i. Implement the backward pass of the convolutional neural network

Next, you are going to implement the backward pass of gradients that stem from the loss function and propogate all the way to the 1st layer of the network. 

As usual, we need to compute several types of gradients for EACH network layer (see instance variable placeholders in the constructor):
- `d_net_act`
- `d_net_wts` (for layers that have weights)
- `d_net_in`
- `d_b` (for layers that have weights)

#### The flow of the backward gradients

- `d_upstream` gives us the gradient from the layer above that GOT US TO the `net_act` stage of the new, current layer. Using `d_upstream`, we compute `d_net_in` via (`backward_netAct_to_netIn` in `Layer`) — this gets us to the `net_in` stage, like usual.
- Using `d_net_in`, we compute [`dprev_net_act`, `d_net_wts`, `d_b`] via `backward_netIn_to_prevLayer_netAct` in `Layer`, where `dprev_net_act` is the net_act gradient for layer beneath the current one (the `d_upstream` for the one layer down).
- `dprev_net_act` becomes `d_upstream` for the next layer down, and the process repeats...

We only need to "store" `d_b` and `d_net_wts` as instance variables in a layer because these are needed for weight updates during training/backprop, the other variables are just needed temporarily as a means to compute `d_b` and `d_net_wts` in areas downstream. 

#### Goal of backward pass

We need to compute these variables (`d_net_act` `d_net_wts` `d_net_in` `d_b`) for each network layer. Start working at the network level and drill down into the layer-specific implementations.

Implement the following:

- `backward` in `Network`
- `backward` in `Layer`
- `backward_netAct_to_netIn` in `Layer` (computes `d_net_in`)
- `backward_netIn_to_prevLayer_netAct` in `Dense` (computes [`dprev_net_act`, `d_net_wts`, `d_b`])
- `backward_netIn_to_prevLayer_netAct` in `Flatten` (computes `dprev_net_act`)
- `backward_netIn_to_prevLayer_netAct` in `MaxPool2D` (arguably the most challenging, so there are more detailed instructions)

(`backward_netIn_to_prevLayer_netAct` in `Conv2D` is already done for you :)

#### 4i. (i) Test backwards thru output (Dense) layer

In [ ]:
rng = np.random.default_rng(3)

n_inputs = 5
n_hidden = 10
n_chans, img_y, img_x = 1, 3, 3
n_units_prev_layer = n_chans*img_y*img_x

# Define test inputs/net quantities
inputs = rng.random((n_inputs, n_chans, img_y, img_x))  # 5, 1, 3, 3
wts = rng.random((n_units_prev_layer, n_hidden))  # 9, 10
b = rng.random((n_hidden,))  # 10
d_upstream = rng.random((n_inputs, n_hidden))  # 5, 10

# Create layer and fill it with the test values
f_layer = Flatten(8, 'Flatten')
dense_layer = Dense(10, 'Output', units=n_hidden, n_units_prev_layer=n_units_prev_layer)
f_layer.input = inputs
f_layer.compute_net_in()
dense_layer.input = f_layer.net_in
dense_layer.wts = wts
dense_layer.b = b
dense_layer.verbose = False  # Toggle this on/off as needed
dense_layer.compute_net_in()
dense_layer.compute_net_act()

# Do the backwards pass thru the layer
dprev_net_act, d_wts, d_b = dense_layer.backward_netIn_to_prevLayer_netAct(d_upstream)
print(f'Shapes: d_b {d_b.shape}, d_wts {d_wts.shape}, and dprev_net_act {dprev_net_act.shape}')
print(f'Shapes should be: d_b (10,), d_wts (9, 10), and dprev_net_act (5, 9)')

print()
print(f'Your d_b is\n{d_b}')
print()
print(f'Your d_wts is\n{d_wts}')
print()
print(f'Your dprev_net_act is\n{dprev_net_act}')
print()

**The above gradients should be:**

    Your d_b is
    [2.939972  1.5288632 1.5934582 2.5569741 3.42369   1.9033922 3.0180793
    2.0253668 3.0044666 3.421824 ]

    Your d_wts is
    [[1.022262  0.4203976 0.7783139 0.9918202 1.5201265 0.7833104 0.9345904
      0.9171647 1.1861436 1.2538697]
    [1.2864636 0.4407391 0.4169773 0.9978163 1.1788296 0.648836  1.1450934
      0.8800866 1.0098424 1.1541508]
    [1.5680805 0.8982268 1.0854391 1.4559452 1.6867111 1.2276175 1.6481592
      1.138053  1.8404759 1.8155044]
    [1.3733557 0.6643123 0.648199  1.187348  1.7787889 0.8814593 1.6552009
      0.9005457 1.3065905 1.8090585]
    [1.5314267 0.637941  0.4818006 1.1545242 1.4317624 0.6647081 1.1984389
      1.0025069 1.2153267 1.3285968]
    [1.7655459 1.050588  0.9767144 1.506954  1.9827554 1.0557073 1.5921494
      1.1796107 1.8556087 1.9352423]
    [2.3887655 1.3817035 1.2739924 2.0335991 2.9047506 1.3827122 2.2102931
      1.5998073 2.4492552 2.7442209]
    [1.460473  0.3908762 0.5608205 1.1835016 1.5073256 0.8063068 1.2858772
      1.1173971 1.1810884 1.3509113]
    [1.614364  0.9954862 0.9571274 1.4355557 1.9242129 1.1056847 1.7704159
      1.05157   1.7707872 2.0211374]]

    Your dprev_net_act is
    [[1.9059042 2.256193  2.3887436 1.8283934 2.1172395 1.4954141 1.9496901
      2.9960784 1.6087471]
    [3.6419362 3.2440551 3.5220766 3.1352212 2.8402101 2.2186481 2.3807716
      3.7879065 2.376959 ]
    [2.2950536 2.2463427 2.4182406 2.3614061 2.0504833 1.5159432 1.8678636
      2.8608441 1.8203996]
    [2.6963963 2.618827  3.0134393 2.6907741 2.7390353 1.8948155 2.5595726
      3.9260347 1.959954 ]
    [3.3462921 3.1563749 3.6705977 2.9847695 3.4139881 2.623484  2.8994443
      4.0945974 2.3683561]]



#### 4i. (ii) Test backwards thru Flatten layer

In [ ]:
rng = np.random.default_rng(1)

n_inputs = 5
n_hidden = 10
n_chans, img_y, img_x = 1, 3, 3
n_units_prev_layer = n_chans*img_y*img_x

# Define test inputs/net quantities
inputs = rng.random((n_inputs, n_chans, img_y, img_x))  # 5, 1, 3, 3
wts = rng.random((n_units_prev_layer, n_hidden))  # 9, 10
b = rng.random((n_hidden,))  # 10
d_upstream = rng.random((n_inputs, n_hidden))  # 5, 10

# Create layer and fill it with the test values
f_layer = Flatten(7, 'Flatten')
dense_layer = Dense(8, 'Output', units=n_hidden, n_units_prev_layer=n_units_prev_layer)
f_layer.input = inputs
f_layer.compute_net_in()
dense_layer.input = f_layer.net_in
dense_layer.wts = wts
dense_layer.b = b
dense_layer.verbose = False  # Toggle this on/off as needed
dense_layer.compute_net_in()
dense_layer.compute_net_act()

# Do the backwards pass thru the layer
dprev_net_act, _, _ = dense_layer.backward_netIn_to_prevLayer_netAct(d_upstream)
dprev_net_act, _, _ = f_layer.backward_netIn_to_prevLayer_netAct(dprev_net_act)
print(f'Shapes: {dprev_net_act.shape=} and it should be (5, 1, 3, 3).')
print(f'The first chunk of your dprev_net_act looks like:\n{dprev_net_act[0,0]}')
print('and it should look like:')
print('''[[3.0649991 2.6812171 2.3775691]
 [2.7365463 1.5570601 2.853221 ]
 [2.2260891 2.4357624 2.2484772]]''')

#### 4i. (iii) Test backwards thru output (MaxPool2D) layer

In [ ]:
rng = np.random.default_rng(0)
n_inputs = 3

# Define test inputs/net quantities
inputs = rng.random((n_inputs, 3, 4, 4))
d_upstream = rng.random((n_inputs, 3, 2, 2))

pool_sz = 2
stride = 2

# Create layer and fill it with the test values
pool_layer = MaxPool2D(1, 'Pool', pool_size=pool_sz, strides=stride)

# Do the forward/backwards pass thru the layer
pool_layer.verbose = False
pool_layer.forward(inputs)
dprev_net_act, _, _ = pool_layer.backward(d_upstream, None)

print(f'Shape: {dprev_net_act.shape} and should be (3, 3, 4, 4).')
print()
print(f'Your d_net_in is\n{dprev_net_act}')

**The above gradients should be:**
    
    Your d_net_in is
    [[[[0.        0.        0.        0.       ]
      [0.        0.4065103 0.        0.909959 ]
      [0.        0.0430669 0.8227063 0.       ]
      [0.        0.        0.        0.       ]]

      [[0.415384  0.        0.        0.       ]
      [0.        0.        0.829804  0.       ]
      [0.        0.        0.3650462 0.       ]
      [0.0099546 0.        0.        0.       ]]

      [[0.        0.        0.        0.       ]
      [0.        0.07863   0.6526146 0.       ]
      [0.        0.        0.        0.       ]
      [0.        0.2738491 0.        0.7026521]]]


    [[[0.        0.        0.1268171 0.       ]
      [0.9438014 0.        0.        0.       ]
      [0.        0.8647783 0.        0.       ]
      [0.        0.        0.        0.0594642]]

      [[0.        0.3807705 0.        0.4297741]
      [0.        0.        0.        0.       ]
      [0.        0.        0.        0.       ]
      [0.        0.4888495 0.9764623 0.       ]]

      [[0.7756912 0.        0.        0.       ]
      [0.        0.        0.        0.3088574]
      [0.        0.        0.        0.       ]
      [0.        0.2698368 0.8631202 0.       ]]]


    [[[0.        0.8813072 0.        0.       ]
      [0.        0.        0.        0.5107065]
      [0.        0.        0.        0.9949173]
      [0.        0.3442957 0.        0.       ]]

      [[0.        0.        0.        0.       ]
      [0.3159435 0.        0.1827124 0.       ]
      [0.        0.8800981 0.8123354 0.       ]
      [0.        0.        0.        0.       ]]

      [[0.        0.        0.9584136 0.       ]
      [0.6678894 0.        0.        0.       ]
      [0.9257146 0.        0.7482485 0.       ]
      [0.        0.        0.        0.       ]]]]

#### 4i. (iv) Test network full backwards pass

(phew)

In [ ]:
rng = np.random.default_rng(10)
n_inputs = 2
X = rng.standard_normal((n_inputs, 3, 32, 32))
y = rng.integers(10, size=n_inputs)

# Do forwards and backwards pass thru network
net = ConvNet4(wt_scale=5e-2, r_seed=1)
loss = net.forward(X, y)
net.backward(y)

# Check various gradients in each layer
print('Output layer')
print('------------------------------------')
print(f'd_wts (1st chunk):\n{net.layers[-1].d_wts[0]}\n')
print(f'd_b (all):\n{net.layers[-1].d_b}\n')
print('------------------------------------')
print('Dense hidden layer')
print('------------------------------------')
print(f'd_wts (1st chunk):\n{net.layers[-2].d_wts[0]}\n')
print(f'd_b (all):\n{net.layers[-2].d_b}\n')
print('------------------------------------')
print('Conv2D layer')
print('------------------------------------')
print(f'd_wts (1st chunk):\n{net.layers[0].d_wts[0,0]}\n')
print(f'd_b (all):\n{net.layers[0].d_b}\n')
print('------------------------------------')

**Above output should be:**

    Output layer
    ------------------------------------
    d_wts (1st chunk):
    [ 0.3431898  0.2133261  0.0844168 -0.3514028  0.3546257  0.0605558
      0.0216805  0.0583535 -0.795683   0.0109376]

    d_b (all):
    [ 0.2210553  0.14336    0.0563526 -0.2734388  0.2212379  0.039703
      0.0146916  0.0373794 -0.4672532  0.006912 ]

    ------------------------------------
    Dense hidden layer
    ------------------------------------
    d_wts (1st chunk):
    [-0.0219397  0.0172228  0.        -0.0235468 -0.0007183 -0.034519
      0.0092343  0.0118275  0.         0.008668   0.         0.
      0.        -0.001092  -0.0177341  0.0127337  0.         0.
      0.0149645  0.         0.0141519  0.0112169  0.0017354  0.0123161
    -0.0047568 -0.0275432  0.0310006 -0.0042568 -0.0007271  0.0076839
      0.0087617  0.0431906  0.0330753  0.         0.0464517  0.0005403
      0.0502644  0.        -0.0269242  0.         0.0059934  0.0253288
      0.0012111  0.        -0.0016052  0.        -0.        -0.0188871
      0.        -0.0258042  0.0094605  0.         0.         0.0067361
    -0.029995   0.         0.0030112  0.        -0.0235838 -0.0260049
      0.0247079  0.0019575  0.         0.        -0.0303252 -0.0280436
      0.0119908  0.         0.00985    0.009503   0.         0.0111796
    -0.         0.        -0.0155056  0.         0.         0.0350763
    -0.         0.         0.        -0.0003382  0.0185528 -0.0117313
      0.0075883  0.0052693  0.        -0.0009278 -0.         0.
      0.0101932  0.0161577  0.0194745 -0.02715    0.0250962  0.
      0.0092954 -0.0123292  0.        -0.013333 ]

    d_b (all):
    [-0.0308205  0.0220939  0.        -0.0313972 -0.0017189 -0.0405281
      0.0141589  0.0181349  0.         0.0149521  0.         0.
      0.        -0.0009321 -0.0271915  0.016989   0.         0.
      0.0229448  0.         0.0181545  0.0171987  0.0022263  0.0188841
    -0.0072935 -0.0355491  0.0383787 -0.0104977 -0.0015551  0.0117816
      0.0134342  0.0575522  0.044813   0.         0.0658683  0.0006932
      0.0644806  0.        -0.0345391  0.         0.0032147  0.0358825
      0.003402   0.        -0.0020592  0.         0.        -0.0298492
      0.        -0.0395652  0.0209744  0.         0.         0.0075217
    -0.0384784  0.         0.0046171  0.        -0.029613  -0.0337412
      0.0338514  0.0030015  0.         0.        -0.0464972 -0.0417379
      0.0190723  0.         0.0094389  0.0096784  0.         0.0171416
      0.         0.        -0.020464   0.         0.         0.0448905
      0.         0.         0.         0.000084   0.0284468 -0.0179875
      0.0111052  0.0067596  0.         0.0017219  0.         0.
      0.0085195  0.0216608  0.0211169 -0.0364758  0.0305229  0.
      0.0102507 -0.0189042  0.        -0.0186998]

    ------------------------------------
    Conv2D layer
    ------------------------------------
    d_wts (1st chunk):
    [[-0.1362295  0.2048112  0.0463401 -0.012283  -0.0902323  0.1238008
      -0.0160383]
    [-0.1575061 -0.3877056  0.0598333  0.0398719  0.059818   0.2381545
      0.0980359]
    [ 0.1334492  0.1679195 -0.0753661 -0.2486824 -0.0594893  0.1021865
      0.2227976]
    [-0.0003345  0.183188   0.0217169 -0.026371  -0.5125119 -0.1761971
      -0.0435054]
    [ 0.0394149  0.0978333 -0.0269406 -0.219169  -0.072041  -0.348753
      -0.0557743]
    [ 0.161513  -0.095408  -0.0703471 -0.0909441  0.0867042  0.1130719
      -0.0096102]
    [-0.2213759 -0.025571   0.1276404 -0.4543047 -0.0171646  0.0149764
      -0.0363872]]

    d_b (all):
    [ 0.1235464  0.1830966 -0.0166502  0.0016563 -0.0115992 -0.2302092
      0.3075932  0.4348645 -0.2880133  0.0893212  0.064857   0.1214334
    -0.1019353  0.0181675  0.3359476 -0.0205121  0.188363   0.1113367
    -0.0180393 -0.2205884 -0.0940726  0.0044058 -0.0456538 -0.0725363
    -0.2119347 -0.1574853 -0.1630172 -0.0391014 -0.2193336 -0.0581874
    -0.0264412  0.2043811]

    ------------------------------------